# Multi-Task Pricer 


Fine-tune an open-source model to predict **both** product category and price from description.

**Format:** `Category: X. Price is $Y.00`

---

## Sections
1. **Setup** — Paths, imports, config
2. **Data prep** — Load items, build multi-task prompts, push to Hub (optional)
3. **Training** — QLoRA fine-tuning (run in Colab with GPU)
4. **Evaluation** — Price MAE + Category accuracy

## 1. Setup

In [ ]:
# pip install -q datasets transformers torch peft bitsandbytes trl accelerate python-dotenv tqdm matplotlib scikit-learn plotly pandas

In [ ]:
import sys
from pathlib import Path

def find_week7_root():
    p = Path.cwd().resolve()
    for _ in range(10):
        if (p / "week7" / "pricer").exists():
            return p / "week7"
        if (p / ".git").exists() and (p / "week7").exists():
            return p / "week7"
        p = p.parent
    return Path.cwd() / "week7"

week7_dir = find_week7_root()
sys.path.insert(0, str(week7_dir))

# Add this folder so we can import multitask_utils
contrib_dir = week7_dir / "community_contributions" / "adams-bolaji"
if contrib_dir.exists():
    sys.path.insert(0, str(contrib_dir))

In [ ]:
import os
import re
from dotenv import load_dotenv
from huggingface_hub import login
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from datasets import Dataset, DatasetDict, load_dataset
from peft import LoraConfig, PeftModel
from trl import SFTTrainer, SFTConfig, DataCollatorForCompletionOnlyLM

from pricer.items import Item
from multitask_utils import (
    build_multitask_prompt,
    build_multitask_completion,
    parse_multitask_completion,
    category_matches,
    extract_price_from_output,
)

load_dotenv(override=True)
if os.environ.get("HF_TOKEN"):
    login(os.environ["HF_TOKEN"], add_to_git_credential=True)
    print("Logged in to HuggingFace")
else:
    print("Set HF_TOKEN for HuggingFace login")

In [ ]:
# Config
LITE_MODE = True
BASE_MODEL = "meta-llama/Llama-3.2-3B"
CUTOFF = 110
HF_USER = "ed-donner"
DATASET_NAME = f"{HF_USER}/items_lite" if LITE_MODE else f"{HF_USER}/items_full"

print(f"Dataset: {DATASET_NAME}")
print(f"Base model: {BASE_MODEL}")

## 2. Data prep

In [ ]:
def item_to_multitask_datapoint(item, tokenizer, max_tokens: int, do_round: bool = True) -> dict:
    summary = getattr(item, "summary", None) or ""
    tokens = tokenizer.encode(summary, add_special_tokens=False)
    if len(tokens) > max_tokens:
        summary = tokenizer.decode(tokens[:max_tokens]).rstrip()
    return {
        "prompt": build_multitask_prompt(summary),
        "completion": build_multitask_completion(
            getattr(item, "category", ""),
            getattr(item, "price", 0),
            do_round,
        ),
    }


train, val, test = Item.from_hub(DATASET_NAME)
print(f"Train: {len(train)}, Val: {len(val)}, Test: {len(test)}")

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

train_dp = [item_to_multitask_datapoint(i, tokenizer, CUTOFF, do_round=True) for i in tqdm(train)]
val_dp = [item_to_multitask_datapoint(i, tokenizer, CUTOFF, do_round=True) for i in tqdm(val)]
test_dp = [item_to_multitask_datapoint(i, tokenizer, CUTOFF, do_round=False) for i in tqdm(test)]

print("Example completion:", train_dp[0]["completion"])

In [ ]:
# Optional: push multi-task dataset to HuggingFace Hub
# MULTITASK_DATASET = "adams-bolaji/items_prompts_multitask_lite"
# if os.environ.get("HF_TOKEN"):
#     DatasetDict({
#         "train": Dataset.from_list(train_dp),
#         "validation": Dataset.from_list(val_dp),
#         "test": Dataset.from_list(test_dp),
#     }).push_to_hub(MULTITASK_DATASET)
#     print(f"Pushed to {MULTITASK_DATASET}")

## 3. Training (QLoRA)

Run this section in **Google Colab** with a GPU (T4 or A100). Set `HF_TOKEN` in Colab Secrets.

In [ ]:
# Uncomment for Colab:
# from google.colab import userdata
# hf_token = userdata.get('HF_TOKEN')
# login(hf_token, add_to_git_credential=True)

In [ ]:
def train_multitask(
    train_data,
    val_data,
    base_model: str = BASE_MODEL,
    output_dir: str = "pricer-multitask",
    hub_model_id: str = None,
    epochs: int = 1,
    batch_size: int = 4,
    lr: float = 5e-5,
    max_seq_len: int = 200,
):
    import torch

    completion_template = "Category: "
    response_template = f"\n{completion_template}"

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )

    model = AutoModelForCausalLM.from_pretrained(
        base_model,
        quantization_config=bnb_config,
        device_map="auto",
    )

    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "up_proj", "down_proj"],
        lora_dropout=0.1,
        bias="none",
        task_type="CAUSAL_LM",
    )

    collator = DataCollatorForCompletionOnlyLM(
        response_template=response_template,
        tokenizer=tokenizer,
        mlm=False,
    )

    train_dataset = Dataset.from_list(train_data)
    eval_dataset = Dataset.from_list(val_data)

    training_args = SFTConfig(
        output_dir=output_dir,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=2,
        learning_rate=lr,
        logging_steps=50,
        save_steps=500,
        eval_strategy="steps",
        eval_steps=500,
        bf16=True,
        hub_model_id=hub_model_id,
        push_to_hub=bool(hub_model_id),
    )

    trainer = SFTTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        processing_class=tokenizer,
        data_collator=collator,
        dataset_text_field="completion",
        max_seq_length=max_seq_len,
    )

    trainer.train()
    if hub_model_id:
        trainer.push_to_hub()
    return trainer

In [ ]:
# Run training (uncomment and adjust for Colab)
TRAIN_SIZE = 2000  # Use subset for quick test
trainer = train_multitask(
    train_dp[:TRAIN_SIZE],
    val_dp[:500],
    output_dir="./pricer-multitask",
    hub_model_id=None,  # Set to "adams-bolaji/pricer-multitask-lite" to push
    epochs=1,
    batch_size=4,
    max_seq_len=200,
)

## 4. Evaluation

In [ ]:
def make_multitask_predictor(model, tokenizer, device="cuda"):
    """Create a predictor that returns (category, price) for an Item."""

    def predict(item):
        summary = getattr(item, "summary", None) or ""
        prompt = build_multitask_prompt(summary)
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        outputs = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
        completion = tokenizer.decode(
            outputs[0][inputs["input_ids"].shape[1]:],
            skip_special_tokens=True,
        )
        return parse_multitask_completion(completion)

    return predict

In [ ]:
def evaluate_multitask(predictor, test_items, size=100):
    """Evaluate both price error and category accuracy."""
    from sklearn.metrics import mean_squared_error, r2_score

    price_guesses, price_truths = [], []
    cat_correct, cat_total = 0, 0

    for i in tqdm(range(min(size, len(test_items)))):
        item = test_items[i]
        cat_pred, price_pred = predictor(item)
        price_truth = item.price
        cat_truth = getattr(item, "category", "")

        price_guesses.append(price_pred)
        price_truths.append(price_truth)
        cat_total += 1
        if category_matches(cat_pred, cat_truth):
            cat_correct += 1

    mae = sum(abs(p - t) for p, t in zip(price_guesses, price_truths)) / len(price_guesses)
    mse = mean_squared_error(price_truths, price_guesses)
    r2 = r2_score(price_truths, price_guesses) * 100
    cat_acc = (cat_correct / cat_total) * 100

    print(f"Price MAE: ${mae:,.2f}")
    print(f"Price MSE: {mse:,.0f}")
    print(f"Price R²: {r2:.1f}%")
    print(f"Category accuracy: {cat_acc:.1f}% ({cat_correct}/{cat_total})")
    return {"mae": mae, "mse": mse, "r2": r2, "cat_acc": cat_acc}

In [ ]:
# Load fine-tuned model and evaluate (set ADAPTER_PATH after training)
ADAPTER_PATH = None  

if ADAPTER_PATH:
    import torch
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL, quantization_config=bnb, device_map="auto"
    )
    model = PeftModel.from_pretrained(model, ADAPTER_PATH)
    predictor = make_multitask_predictor(model, tokenizer)
    evaluate_multitask(predictor, test, size=100)
else:
    print("Set ADAPTER_PATH to your fine-tuned model to run evaluation.")

In [ ]:
# Baseline: zero-shot (no fine-tuning) — load base model only
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, device_map="auto")
predictor = make_multitask_predictor(model, tokenizer)
evaluate_multitask(predictor, test, size=50)